# Phase 1: QLoRA Fine-Tuning

**Runtime:** GPU → Runtime → Change runtime type → T4 GPU

## 0. Install

In [ ]:
#!pip install -q bitsandbytes transformers peft trl accelerate datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.0 MB/s eta 0:00:00


## 1. Imports

In [1]:
#!pip install -q trl
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


## 2. HuggingFace Login

Accept Llama-3.2-3B license first at huggingface.co/meta-llama/Llama-3.2-3B

In [2]:
from huggingface_hub import login
login()  # paste your HF token when prompted

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 3. Load Model in 4-bit (QLoRA)

- NF4 quantization shrinks model from ~6GB to ~1.5GB GPU memory
- Base weights are frozen — we never update them

In [7]:
!pip install -U bitsandbytes>=0.46.1

MODEL_ID = "meta-llama/Llama-3.2-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded. Params: {model.num_parameters():,}")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded. Params: 3,212,749,824


## 4. LoRA Config

- Injects trainable A x B matrices into attention layers
- r=16 is rank — controls adapter capacity
- Only ~1-3% of params will be trainable

In [4]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1-3% trainable

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## 5. Dataset

ultrachat_200k — 200k instruction-following conversations. We use 2000 examples for this learning run.

In [5]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
dataset = dataset.select(range(2000))

# Fix: Explicitly set the chat_template for the tokenizer if it's not already set.
# This ensures that tokenizer.apply_chat_template has a template to use.
# This specific template is based on the Llama-3 instruction format.
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% if messages[0]['role'] == 'system' %}"
        "    {% set loop_messages = messages[1:] %}"
        "    {% set system_message = messages[0]['content'] %}"
        "{% else %}"
        "    {% set loop_messages = messages %}"
        "    {% set system_message = false %}"
        "{% endif %}"
        "{% for message in loop_messages %}"
        "    {% if loop.index == 1 and system_message != false %}"
        "        {% set content = system_message + '\n' + message['content'] %}"
        "    {% else %}"
        "        {% set content = message['content'] %}"
        "    {% endif %}"
        "    {% if message['role'] == 'user' %}"
        "        {{ '<|start_header_id|>user<|end_header_id|>\n' + content + '<|eot_id|>' }}"
        "    {% elif message['role'] == 'assistant' %}"
        "        {{ '<|start_header_id|>assistant<|end_header_id|>\n' + content + '<|eot_id|>' }}"
        "    {% endif %}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "    {{ '<|start_header_id|>assistant<|end_header_id|>\n' }}"
        "{% endif %}"
    )

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
print(dataset[0]["text"][:300])

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-f7dfac4afe5(…):   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

                                    <|start_header_id|>user<|end_header_id|>
These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?
On your Collections pages & Featured Collections sections, you can easi


In [17]:
# Cell A — re-attach adapter
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(type(model))  # must say PeftModelForCausalLM

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
<class 'peft.peft_model.PeftModelForCausalLM'>


## 6. Train

Effective batch size = 2 x 4 = 8. 100 steps ~15-20 min on T4.

In [20]:
training_args = SFTConfig(
    output_dir="./qlora-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    max_steps=100,
    logging_steps=10,
    save_steps=50,
    optim="paged_adamw_8bit",
    bf16=True,          # changed from fp16=True
    gradient_checkpointing=True,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()

Step,Training Loss
10,1.647330
20,1.508421
30,1.479542
40,1.457458
50,1.458218
60,1.412034
70,1.404030
80,1.363284
90,1.436878
100,1.410276


TrainOutput(global_step=100, training_loss=1.457746992111206, metrics={'train_runtime': 1807.3296, 'train_samples_per_second': 0.443, 'train_steps_per_second': 0.055, 'total_flos': 1.3468195233005568e+16, 'train_loss': 1.457746992111206})

## 7. Save Adapter Weights

Only saves LoRA adapter (~50-100MB), not the full base model (~6GB).

In [21]:
model.save_pretrained("./adapter_weights")
tokenizer.save_pretrained("./adapter_weights")
print("Saved.")
!ls -lh adapter_weights/

Saved.
total 63M
-rw-r--r-- 1 root root 1.1K May 10 14:21 adapter_config.json
-rw-r--r-- 1 root root  47M May 10 14:21 adapter_model.safetensors
-rw-r--r-- 1 root root 5.1K May 10 14:21 README.md
-rw-r--r-- 1 root root  335 May 10 14:21 tokenizer_config.json
-rw-r--r-- 1 root root  17M May 10 14:21 tokenizer.json


## 8. Quick Inference Check

In [22]:
model.eval()

prompt = "Explain the difference between supervised and unsupervised learning."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Explain the difference between supervised and unsupervised learning. How does supervised learning work in the context of machine learning and data science? What are some examples of supervised learning tasks in data science applications? Provide a step-by-step guide to implementing supervised learning algorithms in Python. Finally, discuss some potential challenges and limitations of supervised learning and how they can be addressed. Use examples and relevant research to support your answer.

Supervised learning is a machine learning approach that involves providing training data to the algorithm, which then learns to identify patterns in the data that can be used to make predictions about new data. On the other hand, unsupervised learning is a machine learning approach that involves providing data to the algorithm, which then learns to identify patterns in the data that can be used to cluster or group the data into different categories.

In supervised learning, the algorithm is traine

## 9. Download Adapter Weights

In [23]:
!zip -r adapter_weights.zip adapter_weights/
from google.colab import files
files.download("adapter_weights.zip")

  adding: adapter_weights/ (stored 0%)
  adding: adapter_weights/adapter_config.json (deflated 59%)
  adding: adapter_weights/tokenizer.json (deflated 85%)
  adding: adapter_weights/README.md (deflated 65%)
  adding: adapter_weights/tokenizer_config.json (deflated 45%)
  adding: adapter_weights/adapter_model.safetensors (deflated 21%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>